In [1]:
!pip -q install pandas numpy scikit-learn joblib fastapi uvicorn pydantic requests httpx reportlab
import os, json, time, shutil, zipfile, subprocess, sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

BASE_DIR = Path('/content/CLA4_ML_Model_Deployment')
DATA_DIR, OUT_DIR = BASE_DIR/'data', BASE_DIR/'outputs'
REPORT_DIR, SCREEN_DIR = BASE_DIR/'reports', BASE_DIR/'screenshots'
for folder in [DATA_DIR, OUT_DIR, REPORT_DIR, SCREEN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
print('Project folder:', BASE_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.6 MB/s eta 0:00:00
Project folder: /content/CLA4_ML_Model_Deployment


## Dataset — upload the required Kaggle `train.csv`

In [3]:
from google.colab import files
train_path = DATA_DIR/'train.csv'
if not train_path.exists():
    print('Upload the official Titanic train.csv downloaded from Kaggle.')
    uploaded = files.upload()
    for filename, content in uploaded.items():
        # More flexible check: accepts 'train.csv' or variations like 'train-selected-columns.csv'
        if 'train' in filename.lower() and filename.lower().endswith('.csv'):
            train_path.write_bytes(content)
            print(f'Saved uploaded {filename} as train.csv')
            break
if not train_path.exists():
    raise FileNotFoundError('Please upload train.csv from https://www.kaggle.com/competitions/titanic/data')
df = pd.read_csv(train_path)
if 'Survived' not in df.columns:
    raise ValueError('Use the labelled Kaggle train.csv file, which contains the Survived target column.')
print('Dataset shape:', df.shape)
display(df.head())

Upload the official Titanic train.csv downloaded from Kaggle.


Saving train-selected-columns.csv to train-selected-columns (1).csv
Saved uploaded train-selected-columns (1).csv as train.csv
Dataset shape: (891, 10)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500


## Task 1 — Preprocessing, training, evaluation, and saving `model.pkl`

In [5]:
# Dynamically adjust features based on what is available in the uploaded dataset
FEATURES = [col for col in ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'] if col in df.columns]
TARGET = 'Survived'
missing_columns = set([TARGET]) - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing expected Titanic columns: {missing_columns}')

X, y = df[FEATURES].copy(), df[TARGET].astype(int)
numeric_features = [col for col in ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'] if col in FEATURES]
categorical_features = [col for col in ['Sex', 'Embarked'] if col in FEATURES]

# Configure transformers dynamically depending on available columns
transformers = []
if numeric_features:
    transformers.append(('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features))
if categorical_features:
    transformers.append(('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features))

preprocessor = ColumnTransformer(transformers)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
model_path = BASE_DIR/'model.pkl'
joblib.dump(pipeline, model_path)
print(f'Using features: {FEATURES}')
print(f'Test accuracy: {accuracy:.4f}')
print(classification_report(y_test, y_pred, zero_division=0))
print('Saved complete preprocessing + model pipeline:', model_path)
metrics = {'dataset_rows': int(len(df)), 'train_rows': int(len(X_train)),
           'test_rows': int(len(X_test)), 'accuracy': float(accuracy),
           'features': FEATURES, 'model': 'LogisticRegression with preprocessing Pipeline'}
(REPORT_DIR/'training_metrics.json').write_text(json.dumps(metrics, indent=2))
model = joblib.load(model_path)

Using features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
Test accuracy: 0.8045
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       110
           1       0.77      0.71      0.74        69

    accuracy                           0.80       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179

Saved complete preprocessing + model pipeline: /content/CLA4_ML_Model_Deployment/model.pkl


## Task 2 — Online inference (one unseen held-out test record)

In [6]:
online = X_test.iloc[[0]].copy()
online_out = online.copy()
online_out['predicted_survived'] = model.predict(online).astype(int)
display(online_out)
online_out.to_csv(OUT_DIR/'online_prediction.csv', index=False)
print('Saved:', OUT_DIR/'online_prediction.csv')


,Pclass,Sex,Age,SibSp,Parch,Fare,predicted_survived
565,3,male,24.0,2,0,24.15,0


Saved: /content/CLA4_ML_Model_Deployment/outputs/online_prediction.csv


## Task 3 — Batch inference (20 unseen held-out test records)

In [7]:
batch = X_test.iloc[:min(20, len(X_test))].copy()
batch_out = batch.copy()
batch_out['predicted_survived'] = model.predict(batch).astype(int)
# Actual labels are retained only for offline evaluation, never as model inputs.
batch_out['actual_survived_for_evaluation_only'] = y_test.loc[batch.index].to_numpy()
batch_out.to_csv(OUT_DIR/'batch_predictions.csv', index=False)
print('Saved batch_predictions.csv with', len(batch_out), 'records')
display(batch_out)


Saved batch_predictions.csv with 20 records


,Pclass,Sex,Age,SibSp,Parch,Fare,predicted_survived,actual_survived_for_evaluation_only
565,3,male,24.0,2,0,24.1500,0,0
160,3,male,44.0,0,1,16.1000,0,0
553,3,male,22.0,0,0,7.2250,0,1
860,3,male,41.0,2,0,14.1083,0,0
241,3,female,NaN,1,0,15.5000,1,1
559,3,female,36.0,1,0,17.4000,0,1
387,2,female,36.0,0,0,13.0000,1,1
536,1,male,45.0,0,0,26.5500,0,0
698,1,male,49.0,1,1,110.8833,0,0
99,2,male,34.0,1,0,26.0000,0,0


## Task 4 — Streaming inference (10 records, one-second delay)

In [8]:
stream = X_test.iloc[:min(10, len(X_test))]
stream_rows = []
print('Streaming starts; each incoming record is processed separately.')
for number, (_, row) in enumerate(stream.iterrows(), start=1):
    pred = int(model.predict(row.to_frame().T)[0])
    print(f'Record {number}: predicted_survived={pred}')
    stream_rows.append({**row.to_dict(), 'record_number': number, 'predicted_survived': pred})
    time.sleep(1)
pd.DataFrame(stream_rows).to_csv(OUT_DIR/'streaming_predictions.csv', index=False)
print('Saved:', OUT_DIR/'streaming_predictions.csv')
sample = {k: (None if pd.isna(v) else (v.item() if hasattr(v, 'item') else v))
          for k, v in X_test.iloc[0].to_dict().items()}
(BASE_DIR/'sample_request.json').write_text(json.dumps(sample, indent=2))
print('Sample API request:', sample)


Streaming starts; each incoming record is processed separately.
Record 1: predicted_survived=0
Record 2: predicted_survived=0
Record 3: predicted_survived=0
Record 4: predicted_survived=0
Record 5: predicted_survived=1
Record 6: predicted_survived=0
Record 7: predicted_survived=1
Record 8: predicted_survived=0
Record 9: predicted_survived=0
Record 10: predicted_survived=0
Saved: /content/CLA4_ML_Model_Deployment/outputs/streaming_predictions.csv
Sample API request: {'Pclass': 3, 'Sex': 'male', 'Age': 24.0, 'SibSp': 2, 'Parch': 0, 'Fare': 24.15}


## Task 5 — Create the FastAPI application and Python client

In [9]:
app_source = r'''from pathlib import Path
from typing import Optional
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field, field_validator

BASE_DIR = Path(__file__).resolve().parent
MODEL_PATH = BASE_DIR / "model.pkl"
if not MODEL_PATH.exists():
    raise FileNotFoundError("model.pkl not found. Run the notebook training cells first.")
model = joblib.load(MODEL_PATH)
app = FastAPI(title="Titanic Survival Prediction API",
              description="CLA 4 model-serving API", version="1.0.0")

class Passenger(BaseModel):
    Pclass: int = Field(..., ge=1, le=3, description="Passenger class 1, 2, or 3")
    Sex: str
    Age: Optional[float] = Field(default=None, ge=0, le=100)
    SibSp: int = Field(..., ge=0, le=20)
    Parch: int = Field(..., ge=0, le=20)
    Fare: float = Field(..., ge=0)
    Embarked: Optional[str] = None

    @field_validator("Sex")
    @classmethod
    def validate_sex(cls, value):
        value = value.strip().lower()
        if value not in {"male", "female"}:
            raise ValueError("Sex must be male or female")
        return value

    @field_validator("Embarked")
    @classmethod
    def validate_embarked(cls, value):
        if value is None or not value.strip():
            return None
        value = value.strip().upper()
        if value not in {"C", "Q", "S"}:
            raise ValueError("Embarked must be C, Q, S, or null")
        return value

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True}

@app.post("/predict")
def predict(passenger: Passenger):
    data = pd.DataFrame([passenger.model_dump()])
    prediction = int(model.predict(data)[0])
    probabilities = model.predict_proba(data)[0].tolist()
    return {
        "predicted_survived": prediction,
        "label": "Survived" if prediction == 1 else "Did not survive",
        "probability": {
            "did_not_survive": round(float(probabilities[0]), 6),
            "survived": round(float(probabilities[1]), 6)
        }
    }
'''
(BASE_DIR/'app.py').write_text(app_source)

client_source = r'''import json
from pathlib import Path
import requests
BASE_DIR = Path(__file__).resolve().parent
with open(BASE_DIR/'sample_request.json', encoding='utf-8') as f:
    payload = json.load(f)
health = requests.get('http://127.0.0.1:8000/health', timeout=10)
print('GET /health:', health.status_code, health.json())
response = requests.post('http://127.0.0.1:8000/predict', json=payload, timeout=10)
print('POST /predict:', response.status_code, response.json())
response.raise_for_status()
'''
(BASE_DIR/'client.py').write_text(client_source)

requirements = '''pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
joblib>=1.3
fastapi>=0.110
uvicorn[standard]>=0.27
pydantic>=2.0
requests>=2.31
httpx>=0.26
reportlab>=4.0
'''
(BASE_DIR/'requirements.txt').write_text(requirements)
print('Created app.py, client.py, requirements.txt')


Created app.py, client.py, requirements.txt


### Run and test the API in Colab

In [10]:
api_log = open(BASE_DIR/'outputs'/'uvicorn.log', 'w')
api_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app:app', '--host', '127.0.0.1', '--port', '8000'],
    cwd=str(BASE_DIR), stdout=api_log, stderr=subprocess.STDOUT
)
import requests
api_ready = False
for _ in range(30):
    try:
        health_response = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if health_response.ok:
            api_ready = True
            break
    except Exception:
        time.sleep(1)
if not api_ready:
    print((BASE_DIR/'outputs'/'uvicorn.log').read_text()[-4000:])
    raise RuntimeError('FastAPI did not start; inspect uvicorn.log')
print('GET /health:', health_response.status_code, health_response.json())
with open(BASE_DIR/'sample_request.json') as f:
    payload = json.load(f)
valid_response = requests.post('http://127.0.0.1:8000/predict', json=payload, timeout=10)
print('Valid POST /predict:', valid_response.status_code, valid_response.json())
invalid_payload = dict(payload)
invalid_payload.update({'Pclass': 9, 'Sex': 'unknown', 'Age': -5, 'SibSp': -1, 'Fare': -2, 'Embarked': 'X'})
invalid_response = requests.post('http://127.0.0.1:8000/predict', json=invalid_payload, timeout=10)
print('Invalid POST /predict status (expected 422):', invalid_response.status_code)
print(invalid_response.json())
assert valid_response.status_code == 200
assert invalid_response.status_code == 422
print('API valid and invalid tests passed.')
print('For local Swagger UI, run uvicorn app:app --reload and open http://127.0.0.1:8000/docs')


GET /health: 200 {'status': 'ok', 'model_loaded': True}
Valid POST /predict: 200 {'predicted_survived': 0, 'label': 'Did not survive', 'probability': {'did_not_survive': 0.924709, 'survived': 0.075291}}
Invalid POST /predict status (expected 422): 422
{'detail': [{'type': 'less_than_equal', 'loc': ['body', 'Pclass'], 'msg': 'Input should be less than or equal to 3', 'input': 9, 'ctx': {'le': 3}}, {'type': 'value_error', 'loc': ['body', 'Sex'], 'msg': 'Value error, Sex must be male or female', 'input': 'unknown', 'ctx': {'error': {}}}, {'type': 'greater_than_equal', 'loc': ['body', 'Age'], 'msg': 'Input should be greater than or equal to 0', 'input': -5, 'ctx': {'ge': 0.0}}, {'type': 'greater_than_equal', 'loc': ['body', 'SibSp'], 'msg': 'Input should be greater than or equal to 0', 'input': -1, 'ctx': {'ge': 0}}, {'type': 'greater_than_equal', 'loc': ['body', 'Fare'], 'msg': 'Input should be greater than or equal to 0', 'input': -2, 'ctx': {'ge': 0.0}}, {'type': 'value_error', 'loc': [

## Task 6 — Observation answers and PDF report

In [11]:
observations = [
    ("1. Online Inference vs Batch Inference",
     "Online inference predicts for one record when a request arrives and is useful when an immediate answer is needed. Batch inference predicts for many records together, often on a schedule, and is efficient for bulk processing."),
    ("2. Streaming Inference vs Batch Inference",
     "Streaming inference processes incoming records continuously, one at a time (or in small real-time windows), as data arrives. Batch inference waits until a group of records is collected and then processes the group together."),
    ("3. Why FastAPI is used for model serving",
     "FastAPI provides Python route definitions, HTTP request/response handling, automatic OpenAPI documentation and Swagger UI, and Pydantic input validation. It exposes a trained model to other applications through REST endpoints."),
    ("4. Why the same trained model should be reused",
     "Reusing the same saved pipeline keeps predictions consistent across online, batch, streaming, and API inference. It preserves identical preprocessing, avoids unnecessary retraining, and improves reproducibility.")
]
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from xml.sax.saxutils import escape
pdf_path = REPORT_DIR/'observations_report.pdf'
styles = getSampleStyleSheet()
story = [Paragraph('CLA 4: ML Model Deployment Strategies and Model Serving using FastAPI', styles['Title']),
         Spacer(1, 0.2*inch),
         Paragraph('Dataset: Kaggle Titanic — train.csv', styles['Normal']),
         Paragraph(f'Test accuracy from this run: {accuracy:.4f}', styles['Normal']),
         Spacer(1, 0.15*inch)]
for heading, body in observations:
    story.append(Paragraph(escape(heading), styles['Heading2']))
    story.append(Paragraph(escape(body), styles['BodyText']))
    story.append(Spacer(1, 0.12*inch))
SimpleDocTemplate(str(pdf_path), pagesize=A4, rightMargin=0.7*inch, leftMargin=0.7*inch,
                  topMargin=0.7*inch, bottomMargin=0.7*inch).build(story)
(BASE_DIR/'observations.md').write_text('\n\n'.join(f'## {h}\n\n{b}' for h,b in observations))
print('PDF report created:', pdf_path)


PDF report created: /content/CLA4_ML_Model_Deployment/reports/observations_report.pdf
